# Localización subcelular con embeddings de ESM-2

Alternativa a las representaciones "hechas a mano" (composición, dipéptidos, señales):
usamos un **modelo de lenguaje de proteínas (ESM-2)** para convertir cada secuencia en un
vector (embedding) y entrenamos un clasificador lineal encima.

**Corre en CPU** (no hace falta GPU). Usamos el modelo chico `facebook/esm2_t6_8M_UR50D`
(8M parámetros), suficiente para esta tarea y rápido incluso en CPU.

En Colab: `Entorno de ejecución → Cambiar tipo de entorno → CPU` (por defecto).

Datos: subí `features_composicion_senales_fisicoquimicos.csv` cuando la celda lo pida.

In [ ]:
# Colab ya trae torch; solo falta transformers.
!pip -q install transformers

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report

In [ ]:
# Cargar los datos (en Colab: subir el CSV cuando lo pida)
try:
    from google.colab import files
    print("Subí el archivo features_composicion_senales_fisicoquimicos.csv")
    subido = files.upload()
    path = list(subido.keys())[0]
except Exception:
    path = "features_composicion_senales_fisicoquimicos.csv"

df = pd.read_csv(path)
df["localization"].value_counts()

## 1. Embeddings con ESM-2

Tomamos el promedio del último *hidden state* sobre los residuos (sin contar el padding).
Cada proteína pasa a ser un vector de 320 números.

In [ ]:
MODELO = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODELO)
red = AutoModel.from_pretrained(MODELO).eval()

def calcular_embeddings(secuencias, batch=16):
    salida = np.zeros((len(secuencias), red.config.hidden_size), dtype=np.float32)
    for i in range(0, len(secuencias), batch):
        enc = tokenizer(secuencias[i:i + batch], return_tensors="pt",
                        padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            hidden = red(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        salida[i:i + batch] = ((hidden * mask).sum(1) / mask.sum(1)).numpy()
    return salida

X_esm = calcular_embeddings(df["sequence"].tolist())
X_esm.shape

## 2. Comparar con la representación hecha a mano

Mismo split para las dos: 70% entrenamiento / 30% test, estratificado. Métrica: **balanced
accuracy** (accuracy promedio por clase).

In [ ]:
AMINOACIDOS = list("ACDEFGHIKLMNPQRSTVWY")
FISICOQUIMICOS = ["longitud", "peso_molecular", "punto_isoelectrico", "hidrofobicidad_promedio",
                  "aromaticidad", "indice_inestabilidad", "carga_neta_pH7", "fraccion_helice",
                  "fraccion_turn", "fraccion_hoja", "indice_alifatico", "fraccion_cargados",
                  "coef_extincion_molar"]
SENALES = ["senal_peptido", "senal_hidrofobicidad_max", "senal_n_basicos", "nls_monopartita",
           "nls_bipartita", "nls_densidad_basica_10", "nls_densidad_basica_7",
           "nls_carga_neta_nterm20", "tmd_alfa_helices", "tmd_alfa_hidrofobicidad_max",
           "tmd_beta_momento_hidrofobico"]
X_tabla = df[AMINOACIDOS + FISICOQUIMICOS + SENALES].to_numpy(float)
y = df["localization"].to_numpy()

idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3,
                                  stratify=y, random_state=0)

def evaluar(X):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    clf.fit(X[idx_tr], y[idx_tr])
    pred = clf.predict(X[idx_te])
    return balanced_accuracy_score(y[idx_te], pred), pred

ba_tabla, _ = evaluar(X_tabla)
ba_esm, pred_esm = evaluar(X_esm)
print(f"Composición + fisicoquímicos + señales: BA = {ba_tabla:.3f}")
print(f"ESM-2 ({MODELO.split('/')[-1]}):            BA = {ba_esm:.3f}")
print()
print(classification_report(y[idx_te], pred_esm, zero_division=0))

## Para pensar

1. ¿Cuánto mejora ESM-2 sobre la representación hecha a mano? ¿En qué clase se nota más?
2. ESM-2 nunca vio este dataset: ¿por qué igual le sirve? ¿Qué información trae el embedding que
   no capturan la composición ni los dipéptidos?
3. ¿Qué **costo** tiene esta representación comparada con contar aminoácidos? (tiempo de
   cómputo, tamaño del modelo, dependencia de descargar un modelo).
4. ¿Se les ocurre cómo combinar ESM-2 con las señales biológicas (NLS, TMD, péptido señal)?